# T-1 EOD extract — NSE + BSE, full universe

Pulls the exchanges' official **bhavcopy** (UDiFF format) for the most recent trading day and writes CSVs to `~/Desktop/ai-handson/stock-screen/`.

One HTTP request per exchange covers every listed symbol — no API key, no rate limit, no per-symbol loop.

**Baseline caveats** (deliberately not solved here — revisit with the source problem):
- Prices are **unadjusted**. A split shows up as a price crash. Corporate actions are published separately.
- Exchanges occasionally republish revised files, so a re-pull of the same date can differ.
- The browser `User-Agent` below is needed to get past NSE's bot filter. Fine for research; redistribution in a product needs an exchange data licence.

In [ ]:
import csv, io, os, ssl, time, zipfile
import urllib.error, urllib.request
from datetime import date, datetime, timedelta

import pandas as pd

# Many macOS/pyenv Pythons ship without a CA bundle, so urllib raises
# CERTIFICATE_VERIFY_FAILED on hosts curl handles fine. Point at certifi's
# bundle when available rather than disabling verification.
try:
    import certifi
    SSL_CTX = ssl.create_default_context(cafile=certifi.where())
except ImportError:
    SSL_CTX = ssl.create_default_context()

# Colab has /content; locally fall back to the repo folder. Override with
# STOCK_SCREEN_DIR. (Same resolution as 02_indicators.ipynb.)
OUT_DIR     = os.environ.get("STOCK_SCREEN_DIR") or (
    "/content/stock-screen-output" if os.path.isdir("/content")
    else os.path.expanduser("~/Desktop/ai-handson/stock-screen"))
EXCHANGES   = ["NSE", "BSE"]
EQUITY_ONLY = True   # drop bonds, govt secs, ETFs — keep tradable cash equity
TARGET_DATE = None   # None = most recent trading day; else "YYYY-MM-DD"

os.makedirs(OUT_DIR, exist_ok=True)
print("output dir:", OUT_DIR)

output dir: /content/stock-screen-output


## Source config

Both exchanges publish the identical 34-column UDiFF schema, so a single parser handles both.

In [ ]:
UA = ("Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 "
      "(KHTML, like Gecko) Chrome/124.0 Safari/537.36")

SOURCES = {
    "NSE": {
        "url": ("https://nsearchives.nseindia.com/content/cm/"
                "BhavCopy_NSE_CM_0_0_0_{d}_F_0000.csv.zip"),
        "referer": "https://www.nseindia.com/",
        "zipped": True,
        # EQ=equity, BE/BZ=trade-to-trade, SM/ST=SME board
        "equity_series": {"EQ", "BE", "BZ", "SM", "ST"},
    },
    "BSE": {
        "url": ("https://www.bseindia.com/download/BhavCopy/Equity/"
                "BhavCopy_BSE_CM_0_0_0_{d}_F_0000.CSV"),
        "referer": "https://www.bseindia.com/",
        "zipped": False,
        # A/B/T=main board groups, M/MT/MS=SME, X/XT=permitted
        "equity_series": {"A", "B", "T", "M", "MT", "MS", "X", "XT"},
    },
}

OUT_COLS = ["date", "exchange", "symbol", "isin", "series", "name",
            "open", "high", "low", "close", "prev_close", "last",
            "volume", "turnover", "trades"]

## Fetch and parse

Note the content check in `parse()`: BSE's retired URLs answer **HTTP 200 with an HTML error page**, so validating the status code alone silently yields garbage.

In [ ]:
def fetch(url, referer, retries=3):
    """GET with browser headers. Returns bytes, or None if not published."""
    req = urllib.request.Request(url, headers={
        "User-Agent": UA,
        "Referer": referer,
        "Accept": "text/csv,application/zip,*/*",
    })
    for attempt in range(retries):
        try:
            with urllib.request.urlopen(req, timeout=45, context=SSL_CTX) as r:
                return r.read()
        except urllib.error.HTTPError as e:
            if e.code in (403, 404):
                return None                      # holiday / not yet published
            if attempt == retries - 1:
                raise
        except (urllib.error.URLError, TimeoutError):
            if attempt == retries - 1:
                raise
        time.sleep(2 ** attempt)                 # backoff
    return None


def parse(payload, zipped):
    """Bytes -> list of dicts. None if the response isn't actually a bhavcopy."""
    if payload is None:
        return None
    if zipped:
        try:
            zf = zipfile.ZipFile(io.BytesIO(payload))
        except zipfile.BadZipFile:
            return None
        name = next((n for n in zf.namelist() if n.lower().endswith(".csv")), None)
        if name is None:
            return None
        payload = zf.read(name)

    text = payload.decode("utf-8", errors="replace").lstrip("﻿")
    if not text.lstrip().startswith("TradDt"):
        return None                              # HTML error page
    return list(csv.DictReader(io.StringIO(text)))


def normalise(rows, exchange, equity_only):
    keep = SOURCES[exchange]["equity_series"]
    out = []
    for r in rows:
        if r.get("FinInstrmTp") != "STK":         # skip any derivative rows
            continue
        if equity_only and r.get("SctySrs") not in keep:
            continue
        out.append({
            "date":       r["TradDt"],
            "exchange":   exchange,
            "symbol":     r["TckrSymb"],
            "isin":       r["ISIN"],
            "series":     r["SctySrs"],
            "name":       r["FinInstrmNm"],
            "open":       r["OpnPric"],
            "high":       r["HghPric"],
            "low":        r["LwPric"],
            "close":      r["ClsPric"],
            "prev_close": r["PrvsClsgPric"],
            "last":       r["LastPric"],
            "volume":     r["TtlTradgVol"],
            "turnover":   r["TtlTrfVal"],
            "trades":     r["TtlNbOfTxsExctd"],
        })
    return out

## Resolve T-1

"Yesterday" is not necessarily a trading day — weekends and the long list of Indian market holidays both break it. Rather than hardcode a holiday calendar, walk backwards until the exchange actually published a file. That is self-correcting and needs no maintenance.

In [ ]:
def get_day(exchange, day, equity_only=EQUITY_ONLY):
    cfg = SOURCES[exchange]
    rows = parse(fetch(cfg["url"].format(d=day.strftime("%Y%m%d")), cfg["referer"]),
                 cfg["zipped"])
    return None if rows is None else normalise(rows, exchange, equity_only)


def last_trading_day(exchange, start, max_back=10):
    for i in range(max_back):
        day = start - timedelta(days=i)
        if day.weekday() >= 5:                   # Sat/Sun
            continue
        rows = get_day(exchange, day)
        if rows:
            return day, rows
        print(f"  {exchange} {day}: not published (holiday?), stepping back")
    return None, None

## Extract

In [ ]:
start = (datetime.strptime(TARGET_DATE, "%Y-%m-%d").date()
         if TARGET_DATE else date.today() - timedelta(days=1))

frames, resolved = [], {}
for ex in EXCHANGES:
    print(f"{ex}: fetching...")
    if TARGET_DATE:
        day, rows = start, get_day(ex, start)
    else:
        day, rows = last_trading_day(ex, start)
    if not rows:
        print(f"  {ex}: no data found")
        continue
    resolved[ex] = day
    frames.append(pd.DataFrame(rows, columns=OUT_COLS))
    print(f"  {ex} {day}: {len(rows):,} symbols")

assert frames, "no data retrieved from any exchange"

df = pd.concat(frames, ignore_index=True)
num = ["open", "high", "low", "close", "prev_close", "last",
       "volume", "turnover", "trades"]
df[num] = df[num].apply(pd.to_numeric, errors="coerce")
df["date"] = pd.to_datetime(df["date"]).dt.date

if len(set(resolved.values())) > 1:
    print("\nNOTE: exchanges resolved to different dates:",
          {k: str(v) for k, v in resolved.items()})

print(f"\ntotal rows: {len(df):,}")
df.head()

NSE: fetching...
  NSE 2026-08-31: 3,363 symbols
BSE: fetching...
  BSE 2026-08-31: 4,490 symbols

total rows: 7,853


,date,exchange,symbol,isin,series,name,open,high,low,close,prev_close,last,volume,turnover,trades
0,2026-08-31,NSE,20MICRONS,INE144J01027,EQ,20 MICRONS LTD,221.01,227.33,216.50,218.03,222.00,216.50,172302,3.811946e+07,5154
1,2026-08-31,NSE,21STCENMGM,INE253B01015,EQ,21ST CENTURY MGMT SERVICE,40.35,41.84,40.34,40.34,41.16,40.34,2905,1.172337e+05,28
2,2026-08-31,NSE,360ONE,INE466L01038,EQ,360 ONE WAM LIMITED,1199.60,1199.60,1149.30,1175.00,1200.00,1175.00,1412706,1.656558e+09,52727
3,2026-08-31,NSE,GOLD360,INF579M01BB5,EQ,360ONEAMC - GOLD360,153.00,153.00,148.95,149.75,154.70,150.60,25721,3.854417e+06,371
4,2026-08-31,NSE,MSCI360,INF579M01BP5,EQ,360ONEAMC - MSCI360,10.27,10.27,10.08,10.21,10.22,10.21,1079,1.097320e+04,31


## Sanity checks

Structural invariants only — these catch a truncated download, a schema change, or a bad file. They do **not** prove the prices are right.

In [ ]:
traded = df[df["volume"] > 0]

checks = {
    "high >= max(open, close)": (traded["high"] >= traded[["open", "close"]].max(axis=1)).all(),
    "low  <= min(open, close)": (traded["low"]  <= traded[["open", "close"]].min(axis=1)).all(),
    "high >= low":              (traded["high"] >= traded["low"]).all(),
    "no null close":            df["close"].notna().all(),
    "no negative volume":       (df["volume"] >= 0).all(),
    "symbols unique per exch":  not df.duplicated(["exchange", "symbol", "series"]).any(),
    "single trade date":        df["date"].nunique() == 1,
}
for name, ok in checks.items():
    print(f"  {'PASS' if ok else 'FAIL'}  {name}")

print("\nrows per exchange:")
print(df.groupby(["exchange", "series"]).size().unstack(fill_value=0))
print(f"\nzero-volume (untraded) rows: {(df['volume'] == 0).sum():,}")
print(f"symbols on both exchanges (by ISIN): "
      f"{df[df['isin'].notna() & (df['isin'] != '')].groupby('isin')['exchange'].nunique().eq(2).sum():,}")

  PASS  high >= max(open, close)
  PASS  low  <= min(open, close)
  PASS  high >= low
  PASS  no null close
  PASS  no negative volume
  PASS  symbols unique per exch
  PASS  single trade date

rows per exchange:
series      A     B   BE  BZ    EQ    M  MS  MT   SM  ST    T     X   XT
exchange                                                                
BSE       701  1748    0   0     0  281   2  89    0   0  212  1072  385
NSE         0     0  232  37  2647    0   0   0  353  94    0     0    0

zero-volume (untraded) rows: 0
symbols on both exchanges (by ISIN): 2,563


## Write CSVs

In [ ]:
stamp = max(resolved.values()).isoformat()
written = []

for ex, sub in df.groupby("exchange"):
    path = os.path.join(OUT_DIR, f"{ex.lower()}_{stamp}.csv")
    sub.to_csv(path, index=False)
    written.append(path)

combined = os.path.join(OUT_DIR, f"combined_{stamp}.csv")
df.to_csv(combined, index=False)
written.append(combined)

# stable filename for downstream notebooks to import without knowing the date
latest = os.path.join(OUT_DIR, "latest.csv")
df.to_csv(latest, index=False)
written.append(latest)

for p in written:
    print(f"{os.path.getsize(p):>10,} bytes  {p}")

   526,553 bytes  /content/stock-screen-output/bse_2026-08-31.csv
   406,416 bytes  /content/stock-screen-output/nse_2026-08-31.csv
   932,872 bytes  /content/stock-screen-output/combined_2026-08-31.csv
   932,872 bytes  /content/stock-screen-output/latest.csv


## Verify the round-trip

Read back what was written, so the baseline is confirmed on disk rather than only in memory.

In [ ]:
check = pd.read_csv(latest)
print(f"re-read {len(check):,} rows x {len(check.columns)} cols from latest.csv")
print(f"trade date: {check['date'].unique()}")

cols = ["exchange", "symbol", "open", "high", "low", "close", "volume"]
display(check[check.symbol.isin(["RELIANCE", "TCS", "INFY", "HDFCBANK"])][cols])

print("\nmost active by turnover:")
display(check.nlargest(10, "turnover")[["exchange", "symbol", "close", "volume", "turnover"]])

re-read 7,853 rows x 15 cols from latest.csv
trade date: ['2026-08-31']


,exchange,symbol,open,high,low,close,volume
1189,NSE,HDFCBANK,723.05,739.75,704.15,709.00,78157067
1402,NSE,INFY,1138.60,1143.90,1111.10,1133.80,8808120
2462,NSE,RELIANCE,1278.70,1297.60,1271.00,1277.00,34871137
2956,NSE,TCS,2342.00,2399.30,2306.20,2399.30,2905846
3448,BSE,HDFCBANK,721.60,739.50,704.40,709.00,3946627
3464,BSE,INFY,1139.80,1143.65,1111.70,1126.55,168549
3525,BSE,RELIANCE,1280.00,1297.50,1271.05,1285.00,340487
5577,BSE,TCS,2343.90,2364.00,2306.65,2364.00,169592



most active by turnover:


,exchange,symbol,close,volume,turnover
910,NSE,ETERNAL,328.10,272151922,8.897990e+10
1705,NSE,LENSKART,662.40,111714980,7.387435e+10
1691,NSE,LAURUSLABS,1915.00,30995343,5.918953e+10
1189,NSE,HDFCBANK,709.00,78157067,5.634412e+10
54,NSE,ADANIENSOL,1417.40,35589890,5.147929e+10
446,NSE,GROWW,192.06,235495223,4.525717e+10
2462,NSE,RELIANCE,1277.00,34871137,4.458028e+10
55,NSE,ADANIENT,2859.10,12872435,3.756184e+10
279,NSE,ATHERENERG,1717.70,19768043,3.313894e+10
436,NSE,BHARTIARTL,1811.90,16921326,3.126443e+10


In [ ]:
print(f"Listing contents of the output directory: {OUT_DIR}")
!ls -l {OUT_DIR}

Listing contents of the output directory: /content/stock-screen-output
total 2740
-rw-r--r-- 1 root root 526553 Sep  1 03:54 bse_2026-08-31.csv
-rw-r--r-- 1 root root 932872 Sep  1 03:54 combined_2026-08-31.csv
-rw-r--r-- 1 root root 932872 Sep  1 03:54 latest.csv
-rw-r--r-- 1 root root 406416 Sep  1 03:54 nse_2026-08-31.csv


In [ ]:
print(f"Displaying head of {latest} to confirm content:")
display(pd.read_csv(latest).head())

Displaying head of /content/stock-screen-output/latest.csv to confirm content:


,date,exchange,symbol,isin,series,name,open,high,low,close,prev_close,last,volume,turnover,trades
0,2026-08-31,NSE,20MICRONS,INE144J01027,EQ,20 MICRONS LTD,221.01,227.33,216.50,218.03,222.00,216.50,172302,3.811946e+07,5154
1,2026-08-31,NSE,21STCENMGM,INE253B01015,EQ,21ST CENTURY MGMT SERVICE,40.35,41.84,40.34,40.34,41.16,40.34,2905,1.172337e+05,28
2,2026-08-31,NSE,360ONE,INE466L01038,EQ,360 ONE WAM LIMITED,1199.60,1199.60,1149.30,1175.00,1200.00,1175.00,1412706,1.656558e+09,52727
3,2026-08-31,NSE,GOLD360,INF579M01BB5,EQ,360ONEAMC - GOLD360,153.00,153.00,148.95,149.75,154.70,150.60,25721,3.854417e+06,371
4,2026-08-31,NSE,MSCI360,INF579M01BP5,EQ,360ONEAMC - MSCI360,10.27,10.27,10.08,10.21,10.22,10.21,1079,1.097320e+04,31


---
## Baseline established — next steps

`latest.csv` is the screening input. Load it in the next notebook with `pd.read_csv("latest.csv")`.

Deferred to the source problem, in rough priority order:

1. **Corporate actions** — the single biggest correctness gap. Prices here are unadjusted, so any multi-day series breaks across splits and bonuses.
2. **History backfill** — this notebook is one day. Loop the same functions over a date range to build the archive; the exchanges hold years.
3. **Cross-source reconciliation** — diff against a second feed daily and alert on breaks.
4. **Immutable storage** — keep the raw file with a fetch timestamp; exchanges republish revised versions.
5. **Licensing** — required before any of this data leaves the building in a product.